In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/application_train.csv")

df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365

X = df.drop(columns=["TARGET", "SK_ID_CURR"])
y = df["TARGET"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

/var/folders/0k/7s5007550w9b8m22j4485n740000gn/T/ipykernel_70528/2978695676.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
/var/folders/0k/7s5007550w9b8m22j4485n740000gn/T/ipykernel_70528/2978695676.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

features = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "AGE_YEARS",
            "DAYS_EMPLOYED", "DAYS_EMPLOYED_ANOM", "AMT_CREDIT", "AMT_INCOME_TOTAL"]

model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000)),
])
model.fit(X_train[features], y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('impute', ...), ('scale', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3',...,'DAYS_EMPLOYED_ANOM', 'AMT_CREDIT','AMT_INCOME_TOTAL']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missin

In [4]:
proba = model.predict_proba(X_test[features])[:, 1]
proba[:10].round(3)

array([0.081, 0.041, 0.209, 0.098, 0.087, 0.136, 0.019, 0.016, 0.341,
       0.076])

In [5]:
pd.Series(proba).describe()

count    61503.000000
mean         0.080382
std          0.066745
min          0.004057
25%          0.036468
50%          0.058899
75%          0.101496
max          0.652597
dtype: float64

In [6]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test, proba)

0.7233820228813838

In [7]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

for t in [0.5, 0.2, 0.1, 0.08, 0.05]:
    pred = (proba > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"t={t:<5} flagged={pred.sum():>6}  caught={tp:>5}/{tp+fn}  "
          f"recall={recall_score(y_test, pred):.2f}  precision={precision_score(y_test, pred, zero_division=0):.2f}")

t=0.5   flagged=    50  caught=   26/4965  recall=0.01  precision=0.52
t=0.2   flagged=  3653  caught=  995/4965  recall=0.20  precision=0.27
t=0.1   flagged= 15742  caught= 2723/4965  recall=0.55  precision=0.17
t=0.08  flagged= 21642  caught= 3239/4965  recall=0.65  precision=0.15
t=0.05  flagged= 36203  caught= 4155/4965  recall=0.84  precision=0.11


In [8]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, (proba > 0.5).astype(int))

0.9193047493618197

In [9]:
lr = model.named_steps["lr"]
coefs = pd.Series(lr.coef_[0], index=features).sort_values()
coefs

EXT_SOURCE_2         -0.458855
EXT_SOURCE_3         -0.453250
EXT_SOURCE_1         -0.214702
DAYS_EMPLOYED_ANOM   -0.165727
AMT_CREDIT           -0.005338
AMT_INCOME_TOTAL     -0.000221
AGE_YEARS             0.047921
DAYS_EMPLOYED         0.195683
dtype: float64

In [10]:
print("intercept:", lr.intercept_)
np.exp(coefs)   # what each weight means as an odds multiplier

intercept: [-2.70240863]


EXT_SOURCE_2          0.632007
EXT_SOURCE_3          0.635559
EXT_SOURCE_1          0.806782
DAYS_EMPLOYED_ANOM    0.847277
AMT_CREDIT            0.994677
AMT_INCOME_TOTAL      0.999779
AGE_YEARS             1.049088
DAYS_EMPLOYED         1.216142
dtype: float64

In [11]:
from sklearn.tree import DecisionTreeClassifier

# trees can't take blanks either, but they don't need scaling
prep = Pipeline([("impute", SimpleImputer(strategy="median"))])
Xtr = prep.fit_transform(X_train[features])
Xte = prep.transform(X_test[features])

for d in [2, 4, 6, 10, 20, None]:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42).fit(Xtr, y_train)
    tr = roc_auc_score(y_train, tree.predict_proba(Xtr)[:, 1])
    te = roc_auc_score(y_test, tree.predict_proba(Xte)[:, 1])
    print(f"depth={str(d):<5} train AUC={tr:.3f}  test AUC={te:.3f}")

depth=2     train AUC=0.653  test AUC=0.656
depth=4     train AUC=0.702  test AUC=0.701
depth=6     train AUC=0.720  test AUC=0.713
depth=10    train AUC=0.747  test AUC=0.710
depth=20    train AUC=0.910  test AUC=0.579
depth=None  train AUC=1.000  test AUC=0.539


In [13]:
from sklearn.ensemble import RandomForestClassifier

for d in [6, 10, None]:
    rf = RandomForestClassifier(n_estimators=100, max_depth=d, min_samples_leaf=50,
                                n_jobs=-1, random_state=42).fit(Xtr, y_train)
    tr = roc_auc_score(y_train, rf.predict_proba(Xtr)[:, 1])
    te = roc_auc_score(y_test, rf.predict_proba(Xte)[:, 1])
    print(f"depth={str(d):<5} train AUC={tr:.3f}  test AUC={te:.3f}")

depth=6     train AUC=0.731  test AUC=0.727
depth=10    train AUC=0.760  test AUC=0.734
depth=None  train AUC=0.834  test AUC=0.733


In [14]:
# which features does the forest use most?
pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

EXT_SOURCE_2          0.280336
EXT_SOURCE_3          0.263708
AGE_YEARS             0.103143
EXT_SOURCE_1          0.102610
DAYS_EMPLOYED         0.102257
AMT_CREDIT            0.090618
AMT_INCOME_TOTAL      0.051695
DAYS_EMPLOYED_ANOM    0.005634
dtype: float64

In [16]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8,
                    eval_metric="auc", n_jobs=-1, random_state=42)
xgb.fit(X_train[features], y_train)

p_xgb = xgb.predict_proba(X_test[features])[:, 1]
print("train AUC", roc_auc_score(y_train, xgb.predict_proba(X_train[features])[:, 1]))
print("test AUC ", roc_auc_score(y_test, p_xgb))

train AUC 0.7468276266817828
test AUC  0.7411280552177527


In [17]:
num_cols = X_train.select_dtypes("number").columns
xgb_all = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8,
                        eval_metric="auc", n_jobs=-1, random_state=42)
xgb_all.fit(X_train[num_cols], y_train)

print(len(num_cols), "features")
print("test AUC", roc_auc_score(y_test, xgb_all.predict_proba(X_test[num_cols])[:, 1]))

106 features
test AUC 0.7524297358740659


In [18]:
def add_features(d):
    d = d.copy()
    d["CREDIT_INCOME_RATIO"]  = d["AMT_CREDIT"] / d["AMT_INCOME_TOTAL"]
    d["ANNUITY_INCOME_RATIO"] = d["AMT_ANNUITY"] / d["AMT_INCOME_TOTAL"]
    d["CREDIT_TERM"]          = d["AMT_ANNUITY"] / d["AMT_CREDIT"]
    d["EMPLOYED_AGE_RATIO"]   = d["DAYS_EMPLOYED"] / d["DAYS_BIRTH"]
    d["EXT_SOURCE_MEAN"]      = d[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
    return d

X_train2 = add_features(X_train)
X_test2  = add_features(X_test)

# text columns -> category dtype, categories taken from TRAIN only
cat_cols = X_train2.select_dtypes(exclude="number").columns
for c in cat_cols:
    cats = X_train2[c].astype("category").cat.categories
    X_train2[c] = pd.Categorical(X_train2[c], categories=cats)
    X_test2[c]  = pd.Categorical(X_test2[c],  categories=cats)

X_train2.shape

(246008, 127)

In [19]:
xgb2 = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8, enable_categorical=True,
                     eval_metric="auc", n_jobs=-1, random_state=42)
xgb2.fit(X_train2, y_train)
print("train AUC", roc_auc_score(y_train, xgb2.predict_proba(X_train2)[:, 1]))
print("test AUC ", roc_auc_score(y_test, xgb2.predict_proba(X_test2)[:, 1]))

train AUC 0.7743146585635684
test AUC  0.7635359451495999


In [20]:
# did the new features help? top importances
pd.Series(xgb2.feature_importances_, index=X_train2.columns).sort_values(ascending=False).head(12)

EXT_SOURCE_MEAN        0.165573
EXT_SOURCE_3           0.060720
EXT_SOURCE_2           0.049265
NAME_EDUCATION_TYPE    0.027243
DAYS_EMPLOYED          0.027099
APARTMENTS_MEDI        0.026999
FLAG_DOCUMENT_3        0.023890
CODE_GENDER            0.020784
EXT_SOURCE_1           0.018314
NAME_CONTRACT_TYPE     0.018218
CREDIT_TERM            0.017553
LIVINGAREA_AVG         0.016470
dtype: float32